# 🏈 Probabilidades semanales in-season · Temporada 2026

Este notebook actualiza los ratings ELO **partido a partido usando solo
resultados reales ya ocurridos**, y calcula la probabilidad de victoria de
cada juego con el ELO vigente a esa fecha de la temporada. A diferencia del
Notebook 04 (Monte Carlo sobre toda la temporada asumiendo que nada se ha
jugado), aquí la lógica es:

```
Semana 1 -> ELOs de cierre de 2025 + regresión de temporada  (ya lo teníamos)
Semana 2 -> ELOs actualizados con los resultados reales de la semana 1
Semana 3 -> ELOs actualizados con los resultados reales de las semanas 1-2
...
```

`nflreadpy` va agregando resultados reales conforme se juegan los partidos,
así que basta con **volver a correr este notebook cada semana** para obtener
las probabilidades actualizadas de la siguiente semana — sin volver a correr
Monte Carlo completo.

La lógica de ELO se reutiliza de `src/elo.py` (mismas funciones y parámetros
calibrados en el Notebook 02). El motor de este notebook vive en
`src/weekly_probabilities.py`.

In [1]:
# ── §1 · Setup ────────────────────────────────────────────────────────────────
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import nflreadpy as nfl
import plotly.graph_objects as go
import plotly.express as px

from pathlib import Path

from src.weekly_probabilities import (
    build_weekly_snapshots,
    current_week_probabilities,
    team_schedule_view,
)
from src.simulation import TEAM_CONF, TEAM_DIV

# Parámetros calibrados en Notebook 02 (mismos que Notebook 04)
HOME_ADVANTAGE    = 42.2
K_REGULAR         = 40
K_PLAYOFFS        = 48
REGRESSION_FACTOR = 0.40
GLOBAL_MEAN       = 1503.1

SEASON = 2026

# Paleta consistente con src/visualization.py
AFC_COLOR  = '#013369'
NFC_COLOR  = '#D50A0A'
GOLD_COLOR = '#f5c518'

In [2]:
# ── §2 · Cargar datos ─────────────────────────────────────────────────────────
# Calendario de la temporada — nflreadpy trae los marcadores reales conforme
# se van jugando los partidos. Antes de que empiece la temporada, home_score/
# away_score vienen vacíos para todos los juegos.
schedule = nfl.load_schedules(seasons=[SEASON]).to_pandas()

# ELOs finales de la temporada anterior (Notebook 02)
final_elos_df = pd.read_csv('../data/processed/elo_final_2025.csv')
final_elos    = dict(zip(final_elos_df['team'], final_elos_df['elo_final']))

n_played  = schedule[['home_score', 'away_score']].notna().all(axis=1).sum()
n_total   = len(schedule)
print(f"Partidos {SEASON}      : {n_total}")
print(f"Ya jugados          : {n_played}")
print(f"Pendientes           : {n_total - n_played}")
print(f"Equipos con ELO base : {len(final_elos)}")

Partidos 2026      : 272
Ya jugados          : 0
Pendientes           : 272
Equipos con ELO base : 32


In [3]:
# ── §3 · Construir snapshots semanales ────────────────────────────────────────
# games_df        : un renglón por partido, con ELO pre/post y probabilidad
#                    de victoria del local calculada ANTES de que se jugara.
# elo_evolution_df: ELO de cada equipo al inicio de cada semana (para graficar
#                    trayectorias a lo largo de la temporada).
games_df, elo_evolution_df = build_weekly_snapshots(
    schedule_df=schedule,
    final_elos_prev_season=final_elos,
    home_advantage=HOME_ADVANTAGE,
    k_regular=K_REGULAR,
    k_playoffs=K_PLAYOFFS,
    regression_factor=REGRESSION_FACTOR,
    global_mean=GLOBAL_MEAN,
)

print(f"games_df: {games_df.shape}  ·  elo_evolution_df: {elo_evolution_df.shape}")
games_df.head(10)

games_df: (272, 15)  ·  elo_evolution_df: (576, 3)


,season,week,game_type,home_team,away_team,elo_home_pre,elo_away_pre,p_home_pre,p_away_pre,played,home_score,away_score,result,elo_home_post,elo_away_post
0,2026,1,REG,SEA,NE,1643.35,1592.82,0.6304,0.3696,False,NaN,NaN,None,1643.35,1592.82
1,2026,1,REG,LA,SF,1577.37,1559.23,0.5860,0.4140,False,NaN,NaN,None,1577.37,1559.23
2,2026,1,REG,CAR,CHI,1457.76,1531.36,0.4549,0.5451,False,NaN,NaN,None,1457.76,1531.36
3,2026,1,REG,CIN,TB,1467.06,1476.89,0.5465,0.4535,False,NaN,NaN,None,1467.06,1476.89
4,2026,1,REG,DET,NO,1543.71,1451.63,0.6842,0.3158,False,NaN,NaN,None,1543.71,1451.63
5,2026,1,REG,HOU,BUF,1581.66,1579.00,0.5642,0.4358,False,NaN,NaN,None,1581.66,1579.00
6,2026,1,REG,IND,BAL,1472.33,1518.16,0.4948,0.5052,False,NaN,NaN,None,1472.33,1518.16
7,2026,1,REG,JAX,CLE,1545.27,1437.71,0.7031,0.2969,False,NaN,NaN,None,1545.27,1437.71
8,2026,1,REG,PIT,ATL,1521.57,1493.61,0.5996,0.4004,False,NaN,NaN,None,1521.57,1493.61
9,2026,1,REG,TEN,NYJ,1391.90,1401.98,0.5461,0.4539,False,NaN,NaN,None,1391.90,1401.98


In [4]:
# ── §4 · Guardar outputs ───────────────────────────────────────────────────────
out_dir = Path('../data/processed')
weekly_dir = out_dir / 'weekly'
weekly_dir.mkdir(parents=True, exist_ok=True)

games_df.to_csv(out_dir / f'weekly_predictions_{SEASON}.csv', index=False)
elo_evolution_df.to_csv(out_dir / f'elo_evolution_{SEASON}.csv', index=False)

# Un CSV por semana — útil para trackear cómo cambiaron las probabilidades
for week, week_df in games_df.groupby('week'):
    week_df.to_csv(weekly_dir / f'week_{int(week):02d}_probabilities.csv', index=False)

print(f"Guardado: weekly_predictions_{SEASON}.csv, elo_evolution_{SEASON}.csv")
print(f"Guardado: {games_df['week'].nunique()} archivos en data/processed/weekly/")

Guardado: weekly_predictions_2026.csv, elo_evolution_2026.csv
Guardado: 18 archivos en data/processed/weekly/


## §5 · Probabilidades de la próxima semana sin jugar

`current_week_probabilities()` detecta automáticamente la primera semana
pendiente (la próxima por jugarse) y regresa esos partidos con la
probabilidad de victoria calculada usando el ELO más reciente disponible.

In [5]:
next_week_df = current_week_probabilities(games_df)
next_week_n  = int(next_week_df['week'].iloc[0])
print(f"Próxima semana con probabilidades: Semana {next_week_n}")

plot_df = next_week_df.copy()
plot_df['matchup'] = plot_df['away_team'] + ' @ ' + plot_df['home_team']
plot_df = plot_df.sort_values('p_home_pre')
plot_df['home_color'] = plot_df['home_team'].map(TEAM_CONF).map({'AFC': AFC_COLOR, 'NFC': NFC_COLOR})

fig = go.Figure()
fig.add_trace(go.Bar(
    x=plot_df['p_home_pre'],
    y=plot_df['matchup'],
    orientation='h',
    marker_color=plot_df['home_color'],
    text=[f"{p:.0%}" for p in plot_df['p_home_pre']],
    textposition='outside',
    hovertemplate='%{y}<br>Prob. victoria local: %{x:.1%}<extra></extra>',
))
fig.add_vline(x=0.5, line_dash='dash', line_color='gray')
fig.update_layout(
    title=f'Probabilidad de victoria del local · Semana {next_week_n} · {SEASON}',
    xaxis_title='Prob. victoria del equipo local',
    xaxis_tickformat='.0%',
    xaxis_range=[0, 1],
    template='plotly_white',
    height=max(400, 28 * len(plot_df)),
    margin=dict(l=120),
)
fig.show()

Próxima semana con probabilidades: Semana 1


## §6 · Heatmap — probabilidad de victoria por equipo y semana

Vista de toda la temporada: cada celda es la probabilidad de que ese equipo
gane su partido de esa semana (sin importar si jugó de local o visitante),
calculada con el ELO vigente al momento de jugarse (o el más reciente
disponible, para semanas futuras). El hover muestra rival, local/visitante
y si ya se jugó.

In [6]:
team_views = []
for t in sorted(final_elos.keys()):
    v = team_schedule_view(games_df, t)
    v['team'] = t
    team_views.append(v)

long_df = pd.concat(team_views, ignore_index=True)

prob_pivot   = long_df.pivot(index='team', columns='week', values='team_win_prob')
opp_pivot    = long_df.pivot(index='team', columns='week', values='opponent')
home_pivot   = long_df.pivot(index='team', columns='week', values='is_home')
played_pivot = long_df.pivot(index='team', columns='week', values='played')

# Ordenar equipos por conferencia/división para que el heatmap se lea mejor
team_order = sorted(prob_pivot.index, key=lambda t: (TEAM_CONF[t], TEAM_DIV[t], t))
prob_pivot, opp_pivot, home_pivot, played_pivot = (
    df.loc[team_order] for df in (prob_pivot, opp_pivot, home_pivot, played_pivot)
)

customdata = np.dstack([
    opp_pivot.values,
    np.where(home_pivot.values == True, 'Local', 'Visitante'),
    np.where(played_pivot.values == True, 'Jugado', 'Pendiente'),
])

fig = go.Figure(data=go.Heatmap(
    z=prob_pivot.values,
    x=[f'S{w}' for w in prob_pivot.columns],
    y=prob_pivot.index,
    customdata=customdata,
    colorscale='RdBu',
    zmid=0.5,
    zmin=0, zmax=1,
    colorbar=dict(title='Prob.<br>victoria', tickformat='.0%'),
    hovertemplate=(
        '<b>%{y}</b> · %{x}<br>'
        'Rival: %{customdata[0]} (%{customdata[1]})<br>'
        'Prob. victoria: %{z:.1%}<br>'
        '%{customdata[2]}<extra></extra>'
    ),
))
fig.update_layout(
    title=f'Probabilidad de victoria por equipo y semana · {SEASON}',
    template='plotly_white',
    height=900,
    xaxis_title='Semana',
    yaxis=dict(title='', autorange='reversed'),
)
fig.show()

## §7 · Trayectoria de ELO por equipo

Evolución del ELO de cada equipo a lo largo de la temporada, con los
resultados reales ya incorporados. Da clic en un equipo de la leyenda para
aislarlo (doble clic para volver a mostrar todos).

In [ ]:
fig = go.Figure()

for t in sorted(elo_evolution_df['team'].unique()):
    t_df  = elo_evolution_df[elo_evolution_df['team'] == t].sort_values('week')
    color = AFC_COLOR if TEAM_CONF[t] == 'AFC' else NFC_COLOR
    fig.add_trace(go.Scatter(
        x=t_df['week'], y=t_df['elo_pre_week'],
        mode='lines+markers', name=t,
        line=dict(color=color, width=1.5),
        marker=dict(size=4),
        opacity=0.75,
        hovertemplate=f'<b>{t}</b><br>Semana %{{x}}<br>ELO: %{{y:.0f}}<extra></extra>',
    ))

fig.add_hline(y=GLOBAL_MEAN, line_dash='dot', line_color='gray',
              annotation_text=f'Media global ({GLOBAL_MEAN})')

fig.update_layout(
    title=f'Trayectoria de ELO por equipo · {SEASON} (AFC azul · NFC rojo)',
    xaxis_title='Semana', yaxis_title='ELO rating',
    template='plotly_white', height=650,
    legend=dict(font=dict(size=9)),
)
fig.show()

## §8 · Calendario interactivo por equipo

Selecciona un equipo en el menú desplegable para ver su probabilidad de
victoria semana a semana. Verde = ganó, rojo = perdió, gris = pendiente.

In [ ]:
teams_sorted = sorted(final_elos.keys())
fig = go.Figure()

for i, t in enumerate(teams_sorted):
    v = team_schedule_view(games_df, t)
    marker_colors = [
        'gray' if not played else ('#2ecc71' if won == 1 else '#e74c3c')
        for played, won in zip(v['played'], v['team_won'])
    ]
    fig.add_trace(go.Scatter(
        x=v['week'], y=v['team_win_prob'],
        mode='lines+markers',
        name=t,
        visible=(i == 0),
        line=dict(color='#888', width=1),
        marker=dict(size=10, color=marker_colors, line=dict(width=1, color='white')),
        customdata=np.stack([v['opponent'], np.where(v['is_home'], 'Local', 'Visitante')], axis=-1),
        hovertemplate='Semana %{x}<br>Rival: %{customdata[0]} (%{customdata[1]})<br>Prob. victoria: %{y:.1%}<extra></extra>',
    ))

buttons = []
for i, t in enumerate(teams_sorted):
    visible = [j == i for j in range(len(teams_sorted))]
    buttons.append(dict(label=t, method='update',
                         args=[{'visible': visible}, {'title': f'Probabilidad de victoria por semana · {t} · {SEASON}'}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction='down', x=1.15, y=1, showactive=True)],
    title=f'Probabilidad de victoria por semana · {teams_sorted[0]} · {SEASON}',
    xaxis_title='Semana', yaxis_title='Prob. de victoria',
    yaxis=dict(tickformat='.0%', range=[0, 1]),
    template='plotly_white', height=500,
)
fig.add_hline(y=0.5, line_dash='dash', line_color='lightgray')
fig.show()

## Notas de uso

- **Cada semana**: vuelve a correr el notebook completo. `nflreadpy` trae los
  marcadores reales más recientes automáticamente; el ELO se actualiza solo
  con los partidos que ya se jugaron.
- Los CSVs en `data/processed/weekly/` quedan versionados por semana — útil
  para comparar cómo cambió la probabilidad de un juego conforme se acercaba
  (line sharpening) o para armar un post de LinkedIn semana a semana.
- Este motor usa el mismo ELO puro que el Notebook 04 (sin regresión
  logística), consistente con la decisión de diseño documentada en el
  Notebook 03 (`elo_diff` domina el poder predictivo).